
# SmartServe — Data Preprocessing and Feature Engineering

## Business Objective

Transform raw sales and operational data into clean, consistent, and analysis-ready datasets for business intelligence and machine learning.

## Processing Sections

1. Data Loading and Initial Inspection
2. Data Type Standardization
3. Missing Value Treatment
4. Duplicate Handling
5. Data Validation
6. Feature Engineering
7. Machine Learning Dataset Preparation
8. Missing Historical Feature Treatment
9. Export Processed Datasets
10. Final Validation


### Data Loading and Initial Inspection

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_all_data
from src.data_inspection import inspect_all_datasets

sns.set_theme(style="whitegrid")

data = load_all_data()

inspection_report = inspect_all_datasets(data)

display(inspection_report)

print("SMARTSERVE DATA INSPECTION COMPLETED")
print("=" * 50)
print(f"Total datasets: {len(data)}")

2026-11-13 22:15:59,910 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_date.csv
2026-11-13 22:16:00,036 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_store.csv
2026-11-13 22:16:00,045 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_product.csv
2026-11-13 22:16:00,065 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_customer.csv
2026-11-13 22:16:00,103 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_promotion.csv
2026-11-13 22:16:00,112 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_holiday.csv
2026-11-13 22:16:00,117 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartS

,dataset,rows,columns,missing_cells,duplicate_rows,memory_mb
0,dim_date,90,14,86,0,0.03
1,dim_store,5,9,0,0,0.00
2,dim_product,20,10,0,0,0.01
3,dim_customer,1000,6,0,0,0.27
4,dim_promotion,8,7,0,0,0.00
5,dim_holiday,8,4,0,0,0.00
6,dim_weather,450,7,0,0,0.09
7,fact_sales,122318,15,0,0,54.05
8,fact_preparation,9000,8,0,0,1.80
9,fact_inventory,9000,12,0,0,2.01


SMARTSERVE DATA INSPECTION COMPLETED
Total datasets: 11


In [2]:
# 1. Data Loading and Initial Inspection
# Load all raw datasets and inspect their basic structure.

print("Total datasets:", len(data))

for dataset_name, df in data.items():
    print(f"\n{dataset_name}: {df.shape}")

Total datasets: 11

dim_date: (90, 14)

dim_store: (5, 9)

dim_product: (20, 10)

dim_customer: (1000, 6)

dim_promotion: (8, 7)

dim_holiday: (8, 4)

dim_weather: (450, 7)

fact_sales: (122318, 15)

fact_preparation: (9000, 8)

fact_inventory: (9000, 12)

fact_waste: (3648, 8)


In [3]:
# 2. Data Type Standardization
# Standardize column names, convert date columns, and remove duplicate rows.

from src.data_preprocessing import (
    standardize_column_names,
    convert_date_columns,
    remove_duplicate_rows
)

processed_data = {}

for dataset_name, df in data.items():
    df = standardize_column_names(df)
    df = convert_date_columns(df, dataset_name)
    df = remove_duplicate_rows(df, dataset_name)

    processed_data[dataset_name] = df

print("DATA TYPE STANDARDIZATION COMPLETED")
print("=" * 50)

for dataset_name, df in processed_data.items():
    print(f"{dataset_name}: {df.shape}")

2026-11-13 22:20:57,077 | INFO | src.data_preprocessing | dim_date: removed 0 duplicate rows.
2026-11-13 22:20:57,094 | INFO | src.data_preprocessing | dim_store: removed 0 duplicate rows.
2026-11-13 22:20:57,105 | INFO | src.data_preprocessing | dim_product: removed 0 duplicate rows.
2026-11-13 22:20:57,133 | INFO | src.data_preprocessing | dim_customer: removed 0 duplicate rows.
2026-11-13 22:20:57,146 | INFO | src.data_preprocessing | dim_promotion: removed 0 duplicate rows.
2026-11-13 22:20:57,164 | INFO | src.data_preprocessing | dim_holiday: removed 0 duplicate rows.
2026-11-13 22:20:57,176 | INFO | src.data_preprocessing | dim_weather: removed 0 duplicate rows.
2026-11-13 22:20:57,866 | INFO | src.data_preprocessing | fact_sales: removed 0 duplicate rows.
2026-11-13 22:20:57,904 | INFO | src.data_preprocessing | fact_preparation: removed 0 duplicate rows.
2026-11-13 22:20:57,957 | INFO | src.data_preprocessing | fact_inventory: removed 0 duplicate rows.
2026-11-13 22:20:57,979 |

DATA TYPE STANDARDIZATION COMPLETED
dim_date: (90, 14)
dim_store: (5, 9)
dim_product: (20, 10)
dim_customer: (1000, 6)
dim_promotion: (8, 7)
dim_holiday: (8, 4)
dim_weather: (450, 7)
fact_sales: (122318, 15)
fact_preparation: (9000, 8)
fact_inventory: (9000, 12)
fact_waste: (3648, 8)


In [4]:
# 3. Missing Value Analysis
# Identify missing values in all processed datasets.
# Review missing-value counts and percentages before treatment.

from src.data_preprocessing import analyze_missing_values

missing_report = analyze_missing_values(processed_data)

missing_report = missing_report.sort_values(
    by="missing_count",
    ascending=False
)

display(
    missing_report[missing_report["missing_count"] > 0]
)

print("MISSING VALUE ANALYSIS COMPLETED")
print("=" * 50)
print(
    "Total missing values:",
    missing_report["missing_count"].sum()
)

,dataset,column,missing_count,missing_percentage
12,dim_date,holiday_name,86,95.56


MISSING VALUE ANALYSIS COMPLETED
Total missing values: 86


In [5]:
# 4. Missing Value Treatment
# Preserve missing holiday names because they represent non-holiday dates.
# Do not automatically impute other missing values without business justification.

from src.data_preprocessing import handle_missing_values

for dataset_name, df in processed_data.items():
    processed_data[dataset_name] = handle_missing_values(
        df,
        dataset_name
    )

print("MISSING VALUE TREATMENT COMPLETED")
print("=" * 50)

remaining_missing = analyze_missing_values(processed_data)

display(
    remaining_missing[
        remaining_missing["missing_count"] > 0
    ]
)

2026-11-13 22:23:20,718 | INFO | src.data_preprocessing | dim_date: 86 missing values detected.
2026-11-13 22:23:20,729 | INFO | src.data_preprocessing | dim_store: 0 missing values detected.
2026-11-13 22:23:20,742 | INFO | src.data_preprocessing | dim_product: 0 missing values detected.
2026-11-13 22:23:20,756 | INFO | src.data_preprocessing | dim_customer: 0 missing values detected.
2026-11-13 22:23:20,764 | INFO | src.data_preprocessing | dim_promotion: 0 missing values detected.
2026-11-13 22:23:20,769 | INFO | src.data_preprocessing | dim_holiday: 0 missing values detected.
2026-11-13 22:23:20,778 | INFO | src.data_preprocessing | dim_weather: 0 missing values detected.
2026-11-13 22:23:20,901 | INFO | src.data_preprocessing | fact_sales: 0 missing values detected.
2026-11-13 22:23:20,918 | INFO | src.data_preprocessing | fact_preparation: 0 missing values detected.
2026-11-13 22:23:20,929 | INFO | src.data_preprocessing | fact_inventory: 0 missing values detected.
2026-11-13 22:

MISSING VALUE TREATMENT COMPLETED


,dataset,column,missing_count,missing_percentage
12,dim_date,holiday_name,86,95.56


In [6]:
# 5. Duplicate Record Validation
# Check all processed datasets for duplicate records.
# Duplicate rows were removed during standardization.

duplicate_report = []

for dataset_name, df in processed_data.items():
    duplicate_report.append({
        "dataset": dataset_name,
        "duplicate_rows": int(df.duplicated().sum())
    })

duplicate_report = pd.DataFrame(duplicate_report)

display(duplicate_report)

print("DUPLICATE VALIDATION COMPLETED")
print("Total duplicate rows:", duplicate_report["duplicate_rows"].sum())

,dataset,duplicate_rows
0,dim_date,0
1,dim_store,0
2,dim_product,0
3,dim_customer,0
4,dim_promotion,0
5,dim_holiday,0
6,dim_weather,0
7,fact_sales,0
8,fact_preparation,0
9,fact_inventory,0


DUPLICATE VALIDATION COMPLETED
Total duplicate rows: 0


In [7]:
# 6. Numerical Data Validation
# Check numerical columns for invalid negative values.
# Quantities, prices, costs, and other non-negative measures
# should not contain negative values.

from src.data_validation_rules import validate_non_negative_values

numerical_validation_report = validate_non_negative_values(
    processed_data
)

display(numerical_validation_report)

print("NUMERICAL VALIDATION COMPLETED")

2026-11-13 22:24:57,018 | INFO | src.data_validation_rules | Numerical validation completed. Issues found: 0


,dataset,column,invalid_count


NUMERICAL VALIDATION COMPLETED


In [8]:
# 7. Business Rule Validation
# Check important business rules across sales and operational datasets.
# Identify invalid quantities, prices, preparation values, inventory,
# and food waste records.

from src.business_validation import validate_business_rules

business_validation_report = validate_business_rules(processed_data)

display(business_validation_report)

print("BUSINESS RULE VALIDATION COMPLETED")

2026-11-13 22:25:36,136 | INFO | src.business_validation | Business rule validation completed. Issues found: 0


,dataset,business_rule,invalid_count


BUSINESS RULE VALIDATION COMPLETED


In [9]:
# 8. Sales Feature Engineering
# Enrich sales transactions with calendar, store, product,
# customer, promotion, and weather attributes.

from src.feature_engineering import (
    add_calendar_features,
    add_product_store_features,
    add_customer_features,
    add_promotion_features,
    add_weather_features
)

sales_features = add_calendar_features(
    processed_data["fact_sales"],
    processed_data["dim_date"]
)

sales_features = add_product_store_features(
    sales_features,
    processed_data["dim_store"],
    processed_data["dim_product"]
)

sales_features = add_customer_features(
    sales_features,
    processed_data["dim_customer"]
)

sales_features = add_promotion_features(
    sales_features,
    processed_data["dim_promotion"]
)

sales_features = add_weather_features(
    sales_features,
    processed_data["dim_weather"]
)

print("SALES FEATURE ENGINEERING COMPLETED")
print("Original sales shape:", processed_data["fact_sales"].shape)
print("Featured sales shape:", sales_features.shape)

display(sales_features.head())

2026-11-13 22:26:26,293 | INFO | src.feature_engineering | Calendar features added to sales dataset. Rows: 122318 | Columns: 29
2026-11-13 22:26:26,682 | INFO | src.feature_engineering | Product and store features added. Rows: 122318 | Columns: 46
2026-11-13 22:26:26,941 | INFO | src.feature_engineering | Customer features added to sales dataset.
2026-11-13 22:26:27,259 | INFO | src.feature_engineering | Promotion features added | Rows: 122318 | Unmatched promotion IDs: 0
2026-11-13 22:26:27,477 | INFO | src.feature_engineering | Weather features added. Rows: 122318, unmatched records: 0


SALES FEATURE ENGINEERING COMPLETED
Original sales shape: (122318, 15)
Featured sales shape: (122318, 62)


,sales_id,order_id,date_id,store_id,customer_id,product_id,promotion_id,quantity,unit_price,discount_amount,...,promotion_promotion_type,promotion_discount_percent,promotion_start_date,promotion_end_date,promotion_target_category,weather_weather_id,weather_temperature,weather_rainfall,weather_humidity,weather_weather_condition
0,SAL00000001,ORD00000001,20250101,ST001,CUS00522,PRD001,PROMO007,3,120.0,36.0,...,Customer,10,2025-01-01,2025-03-31,All,WTH20250101ST001,23.0,0.0,48,Sunny
1,SAL00000002,ORD00000002,20250101,ST001,CUS00094,PRD001,PROMO007,2,120.0,24.0,...,Customer,10,2025-01-01,2025-03-31,All,WTH20250101ST001,23.0,0.0,48,Sunny
2,SAL00000003,ORD00000003,20250101,ST001,CUS00613,PRD001,PROMO007,1,120.0,12.0,...,Customer,10,2025-01-01,2025-03-31,All,WTH20250101ST001,23.0,0.0,48,Sunny
3,SAL00000004,ORD00000004,20250101,ST001,CUS00556,PRD001,PROMO007,1,120.0,12.0,...,Customer,10,2025-01-01,2025-03-31,All,WTH20250101ST001,23.0,0.0,48,Sunny
4,SAL00000005,ORD00000005,20250101,ST001,CUS00026,PRD001,PROMO007,4,120.0,48.0,...,Customer,10,2025-01-01,2025-03-31,All,WTH20250101ST001,23.0,0.0,48,Sunny


In [15]:
# 9. Sales Feature Validation

from src.feature_engineering import validate_featured_sales

sales_feature_validation = validate_featured_sales(
    processed_data["fact_sales"],
    sales_features
)

print("SALES FEATURE VALIDATION COMPLETED")
display(sales_feature_validation)

print(
    "Passed:",
    sales_feature_validation["passed"].sum(),
    "/",
    len(sales_feature_validation)
)

2026-11-13 22:35:36,202 | INFO | src.feature_engineering | Featured sales validation completed. Passed: 8/8


SALES FEATURE VALIDATION COMPLETED


,check,passed,details
0,Sales row count preserved,True,"Original: 122318, Featured: 122318"
1,Sales IDs preserved in original order,True,Original sales IDs compared with featured sale...
2,No duplicate sales IDs,True,Duplicate IDs: 0
3,customer_ features available,True,Found 6 columns.
4,product_ features available,True,Found 10 columns.
5,store_ features available,True,Found 9 columns.
6,promotion_ features available,True,Found 7 columns.
7,weather_ features available,True,Found 5 columns.


Passed: 8 / 8


In [ ]:
# 10. Daily Demand Dataset Creation
# Aggregate sales transactions into daily product-level demand.
# Combine daily sales with inventory information.

from src.feature_engineering import create_daily_demand_dataset

daily_demand = create_daily_demand_dataset(
    sales_features,
    processed_data["fact_inventory"]
)

print("DAILY DEMAND DATASET CREATED")
print("Dataset shape:", daily_demand.shape)

display(daily_demand.head())
print("\nColumns:")
print(daily_demand.columns.tolist())

In [11]:
# 10. Weather Feature Investigation
# Inspect the actual weather-related columns in the featured sales dataset.

weather_columns = [
    column for column in sales_features.columns
    if "weather" in column.lower()
]

print("Weather-related columns found:", len(weather_columns))
print(weather_columns)

print("\nAll featured sales columns:")
print(sales_features.columns.tolist())

Weather-related columns found: 5
['weather_weather_id', 'weather_temperature', 'weather_rainfall', 'weather_humidity', 'weather_weather_condition']

All featured sales columns:
['sales_id', 'order_id', 'date_id', 'store_id', 'customer_id', 'product_id', 'promotion_id', 'quantity', 'unit_price', 'discount_amount', 'gross_sales', 'net_sales', 'cost_amount', 'profit_amount', 'payment_method', 'full_date', 'year', 'quarter', 'month', 'month_name', 'day_name', 'is_weekend', 'is_holiday', 'season', 'day_of_month', 'day_of_week', 'week_of_year', 'is_month_start', 'is_month_end', 'store_store_name', 'store_city', 'store_area', 'store_store_type', 'store_opening_date', 'store_seating_capacity', 'store_average_daily_customers', 'store_manager_region', 'product_product_name', 'product_category', 'product_subcategory', 'product_size', 'product_unit_cost', 'product_selling_price', 'product_shelf_life_hours', 'product_preparation_time', 'product_active_flag', 'customer_signup_date', 'customer_custom

In [12]:
# 11. Inspect Sales Validation Function
# Review the validation logic to identify why weather features fail.

import inspect
from src.feature_engineering import validate_featured_sales

print(inspect.getsource(validate_featured_sales))

def validate_featured_sales(
    original_sales: pd.DataFrame,
    featured_sales: pd.DataFrame
) -> pd.DataFrame:
    """
    Validate the enriched sales dataset.
    """

    checks = []

    def add_check(check_name, passed, details):
        checks.append({
            "check": check_name,
            "passed": bool(passed),
            "details": details
        })

    # Check 1: Row count preservation
    row_count_ok = len(original_sales) == len(featured_sales)

    add_check(
        "Sales row count preserved",
        row_count_ok,
        f"Original: {len(original_sales)}, "
        f"Featured: {len(featured_sales)}"
    )

    # Check 2: Sales ID preservation
    if "sales_id" in original_sales.columns:
        sales_id_ok = (
            "sales_id" in featured_sales.columns
            and original_sales["sales_id"].equals(
                featured_sales["sales_id"]
            )
        )

        add_check(
            "Sales IDs preserved in original order",
          

In [13]:
# 12. Display Complete Validation Function
# Print the full function so we can identify the incorrect weather check.

import inspect
from src.feature_engineering import validate_featured_sales

source = inspect.getsource(validate_featured_sales)
print(source)

def validate_featured_sales(
    original_sales: pd.DataFrame,
    featured_sales: pd.DataFrame
) -> pd.DataFrame:
    """
    Validate the enriched sales dataset.
    """

    checks = []

    def add_check(check_name, passed, details):
        checks.append({
            "check": check_name,
            "passed": bool(passed),
            "details": details
        })

    # Check 1: Row count preservation
    row_count_ok = len(original_sales) == len(featured_sales)

    add_check(
        "Sales row count preserved",
        row_count_ok,
        f"Original: {len(original_sales)}, "
        f"Featured: {len(featured_sales)}"
    )

    # Check 2: Sales ID preservation
    if "sales_id" in original_sales.columns:
        sales_id_ok = (
            "sales_id" in featured_sales.columns
            and original_sales["sales_id"].equals(
                featured_sales["sales_id"]
            )
        )

        add_check(
            "Sales IDs preserved in original order",
          

In [16]:
# 13. Daily Demand Dataset Creation
# Aggregate sales transactions into daily demand by store and product.
# Combine sales demand with inventory information.

from src.feature_engineering import create_daily_demand_dataset

daily_demand = create_daily_demand_dataset(
    sales_features,
    processed_data["fact_inventory"]
)

print("DAILY DEMAND DATASET CREATED")
print("=" * 50)

print("Dataset shape:", daily_demand.shape)

display(daily_demand.head())

print("\nDataset columns:")
print(daily_demand.columns.tolist())

2026-11-13 22:41:38,575 | INFO | src.feature_engineering | Daily demand dataset created | Rows: 9000 | Columns: 5


DAILY DEMAND DATASET CREATED
Dataset shape: (9000, 5)


,date_id,store_id,product_id,sales_qty,daily_orders
0,20250101,ST001,PRD001,48,20
1,20250102,ST001,PRD001,45,19
2,20250103,ST001,PRD001,77,30
3,20250104,ST001,PRD001,92,35
4,20250105,ST001,PRD001,75,33



Dataset columns:
['date_id', 'store_id', 'product_id', 'sales_qty', 'daily_orders']


In [17]:
# 14. Daily Demand Context Feature Engineering
# Enrich daily demand with calendar, store, product,
# and weather attributes.

from src.feature_engineering import add_demand_context_features

demand_features = add_demand_context_features(
    daily_demand,
    processed_data["dim_date"],
    processed_data["dim_store"],
    processed_data["dim_product"],
    processed_data["dim_weather"]
)

print("DAILY DEMAND CONTEXT FEATURES COMPLETED")
print("=" * 50)

print("Dataset shape:", demand_features.shape)

display(demand_features.head())

print("\nDataset columns:")
print(demand_features.columns.tolist())

2026-11-13 22:42:25,812 | INFO | src.feature_engineering | Demand context features added | Rows: 9000 | Columns: 40


DAILY DEMAND CONTEXT FEATURES COMPLETED
Dataset shape: (9000, 40)


,date_id,store_id,product_id,sales_qty,daily_orders,calendar_full_date,calendar_year,calendar_quarter,calendar_month,calendar_month_name,...,product_unit_cost,product_selling_price,product_shelf_life_hours,product_preparation_time,product_active_flag,weather_weather_id,weather_temperature,weather_rainfall,weather_humidity,weather_weather_condition
0,20250101,ST001,PRD001,48,20,2025-01-01,2025,1,1,January,...,45,120,4,3,True,WTH20250101ST001,23.0,0.0,48,Sunny
1,20250102,ST001,PRD001,45,19,2025-01-02,2025,1,1,January,...,45,120,4,3,True,WTH20250102ST001,23.9,3.4,74,Cloudy
2,20250103,ST001,PRD001,77,30,2025-01-03,2025,1,1,January,...,45,120,4,3,True,WTH20250103ST001,22.6,5.8,77,Cloudy
3,20250104,ST001,PRD001,92,35,2025-01-04,2025,1,1,January,...,45,120,4,3,True,WTH20250104ST001,25.4,3.0,74,Cloudy
4,20250105,ST001,PRD001,75,33,2025-01-05,2025,1,1,January,...,45,120,4,3,True,WTH20250105ST001,23.0,2.1,76,Partly Cloudy



Dataset columns:
['date_id', 'store_id', 'product_id', 'sales_qty', 'daily_orders', 'calendar_full_date', 'calendar_year', 'calendar_quarter', 'calendar_month', 'calendar_month_name', 'calendar_week', 'calendar_day', 'calendar_day_name', 'calendar_day_of_week', 'calendar_is_weekend', 'calendar_is_holiday', 'calendar_holiday_name', 'calendar_season', 'store_store_name', 'store_city', 'store_area', 'store_store_type', 'store_opening_date', 'store_seating_capacity', 'store_average_daily_customers', 'store_manager_region', 'product_product_name', 'product_category', 'product_subcategory', 'product_size', 'product_unit_cost', 'product_selling_price', 'product_shelf_life_hours', 'product_preparation_time', 'product_active_flag', 'weather_weather_id', 'weather_temperature', 'weather_rainfall', 'weather_humidity', 'weather_weather_condition']


In [18]:
# 15. Demand Lag Feature Engineering
# Create historical demand features for each store-product combination.
# Use previous demand observations to support forecasting.

from src.feature_engineering import add_demand_lag_features

demand_features = add_demand_lag_features(demand_features)

print("DEMAND LAG FEATURE ENGINEERING COMPLETED")
print("=" * 50)

print("Dataset shape:", demand_features.shape)

lag_columns = [
    column for column in demand_features.columns
    if "lag" in column.lower() or "rolling" in column.lower()
]

print("\nLag and rolling features:")
print(lag_columns)

display(demand_features.head(10))

2026-11-13 22:44:05,851 | INFO | src.feature_engineering | Demand lag features created | Rows: 9000 | Columns: 47


DEMAND LAG FEATURE ENGINEERING COMPLETED
Dataset shape: (9000, 47)

Lag and rolling features:
['product_active_flag', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28']


,date_id,store_id,product_id,sales_qty,daily_orders,calendar_full_date,calendar_year,calendar_quarter,calendar_month,calendar_month_name,...,weather_rainfall,weather_humidity,weather_weather_condition,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28
0,20250101,ST001,PRD001,48,20,2025-01-01,2025,1,1,January,...,0.0,48,Sunny,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20250102,ST001,PRD001,45,19,2025-01-02,2025,1,1,January,...,3.4,74,Cloudy,48.0,NaN,NaN,NaN,48.000000,48.000000,48.000000
2,20250103,ST001,PRD001,77,30,2025-01-03,2025,1,1,January,...,5.8,77,Cloudy,45.0,NaN,NaN,NaN,46.500000,46.500000,46.500000
3,20250104,ST001,PRD001,92,35,2025-01-04,2025,1,1,January,...,3.0,74,Cloudy,77.0,NaN,NaN,NaN,56.666667,56.666667,56.666667
4,20250105,ST001,PRD001,75,33,2025-01-05,2025,1,1,January,...,2.1,76,Partly Cloudy,92.0,NaN,NaN,NaN,65.500000,65.500000,65.500000
5,20250106,ST001,PRD001,77,31,2025-01-06,2025,1,1,January,...,0.0,44,Sunny,75.0,NaN,NaN,NaN,67.400000,67.400000,67.400000
6,20250107,ST001,PRD001,37,14,2025-01-07,2025,1,1,January,...,0.0,42,Sunny,77.0,NaN,NaN,NaN,69.000000,69.000000,69.000000
7,20250108,ST001,PRD001,60,30,2025-01-08,2025,1,1,January,...,0.0,63,Sunny,37.0,48.0,NaN,NaN,64.428571,64.428571,64.428571
8,20250109,ST001,PRD001,88,37,2025-01-09,2025,1,1,January,...,0.0,60,Partly Cloudy,60.0,45.0,NaN,NaN,66.142857,63.875000,63.875000
9,20250110,ST001,PRD001,59,22,2025-01-10,2025,1,1,January,...,0.0,61,Sunny,88.0,77.0,NaN,NaN,72.285714,66.555556,66.555556


In [19]:
# 16. Demand Lag Feature Validation
# Validate historical demand features and check for possible data leakage.

from src.feature_engineering import validate_demand_lag_features

lag_validation_report = validate_demand_lag_features(
    demand_features
)

print("DEMAND LAG VALIDATION COMPLETED")
print("=" * 50)

display(lag_validation_report)

2026-11-13 22:44:52,640 | INFO | src.feature_engineering | Demand lag validation completed | Passed: 3/3


DEMAND LAG VALIDATION COMPLETED


,check,passed,details
0,No duplicate date-store-product combinations,True,Duplicates: 0
1,Lag-1 uses previous day's demand,True,Compared actual lag-1 values with shifted sale...
2,7-day rolling average excludes current-day demand,True,Compared with shifted 7-day rolling averages.


In [20]:
# 17. Demand Model Dataset Preparation
# Select the demand target and relevant predictive features.
# Exclude identifiers and current-day order information to prevent leakage.

from src.feature_engineering import prepare_demand_model_dataset

demand_model_data = prepare_demand_model_dataset(
    demand_features
)

print("DEMAND MODEL DATASET PREPARATION COMPLETED")
print("=" * 50)

print("Dataset shape:", demand_model_data.shape)

display(demand_model_data.head())

print("\nDataset columns:")
print(demand_model_data.columns.tolist())

print("\nTarget variable: sales_qty")

2026-11-13 22:45:28,972 | INFO | src.feature_engineering | Demand model dataset prepared | Rows: 9000 | Predictors: 43


DEMAND MODEL DATASET PREPARATION COMPLETED
Dataset shape: (9000, 46)


,date_id,calendar_full_date,store_id,product_id,sales_qty,calendar_year,calendar_quarter,calendar_month,calendar_month_name,calendar_week,...,weather_rainfall,weather_humidity,weather_weather_condition,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28
0,20250101,2025-01-01,ST001,PRD001,48,2025,1,1,January,1,...,0.0,48,Sunny,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20250102,2025-01-02,ST001,PRD001,45,2025,1,1,January,1,...,3.4,74,Cloudy,48.0,NaN,NaN,NaN,48.000000,48.000000,48.000000
2,20250103,2025-01-03,ST001,PRD001,77,2025,1,1,January,1,...,5.8,77,Cloudy,45.0,NaN,NaN,NaN,46.500000,46.500000,46.500000
3,20250104,2025-01-04,ST001,PRD001,92,2025,1,1,January,1,...,3.0,74,Cloudy,77.0,NaN,NaN,NaN,56.666667,56.666667,56.666667
4,20250105,2025-01-05,ST001,PRD001,75,2025,1,1,January,1,...,2.1,76,Partly Cloudy,92.0,NaN,NaN,NaN,65.500000,65.500000,65.500000



Dataset columns:
['date_id', 'calendar_full_date', 'store_id', 'product_id', 'sales_qty', 'calendar_year', 'calendar_quarter', 'calendar_month', 'calendar_month_name', 'calendar_week', 'calendar_day', 'calendar_day_name', 'calendar_day_of_week', 'calendar_is_weekend', 'calendar_is_holiday', 'calendar_holiday_name', 'calendar_season', 'store_store_name', 'store_city', 'store_area', 'store_store_type', 'store_opening_date', 'store_seating_capacity', 'store_average_daily_customers', 'store_manager_region', 'product_product_name', 'product_category', 'product_subcategory', 'product_size', 'product_unit_cost', 'product_selling_price', 'product_shelf_life_hours', 'product_preparation_time', 'product_active_flag', 'weather_weather_id', 'weather_temperature', 'weather_rainfall', 'weather_humidity', 'weather_weather_condition', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28']

Target variable: sales_qty


In [21]:
# 18. Chronological Train-Test Split
# Split demand data chronologically to prevent future information
# from entering the training dataset.

from src.feature_engineering import chronological_train_test_split

train_data, test_data = chronological_train_test_split(
    demand_model_data,
    test_size=0.2
)

print("CHRONOLOGICAL TRAIN-TEST SPLIT COMPLETED")
print("=" * 50)

print("Training shape:", train_data.shape)
print("Testing shape:", test_data.shape)

print("\nTraining date range:")
print(train_data["calendar_full_date"].min())
print("to")
print(train_data["calendar_full_date"].max())

print("\nTesting date range:")
print(test_data["calendar_full_date"].min())
print("to")
print(test_data["calendar_full_date"].max())

2026-11-13 22:46:07,728 | INFO | src.feature_engineering | Chronological split completed | Training rows: 7200 | Testing rows: 1800


CHRONOLOGICAL TRAIN-TEST SPLIT COMPLETED
Training shape: (7200, 46)
Testing shape: (1800, 46)

Training date range:
2025-01-01 00:00:00
to
2025-03-13 00:00:00

Testing date range:
2025-03-14 00:00:00
to
2025-03-31 00:00:00


In [22]:
# 19. Feature and Target Separation
# Separate sales_qty as the prediction target.
# Exclude identifiers and date columns from model predictors.

TARGET_COLUMN = "sales_qty"

ID_COLUMNS = [
    "date_id",
    "calendar_full_date",
    "store_id",
    "product_id"
]

feature_columns = [
    column for column in train_data.columns
    if column not in ID_COLUMNS + [TARGET_COLUMN]
]

X_train = train_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()

y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()

print("FEATURE AND TARGET SEPARATION COMPLETED")
print("=" * 50)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nNumber of predictor columns:", len(feature_columns))
print("\nPredictor columns:")
print(feature_columns)

FEATURE AND TARGET SEPARATION COMPLETED
Training features: (7200, 41)
Testing features: (1800, 41)
Training target: (7200,)
Testing target: (1800,)

Number of predictor columns: 41

Predictor columns:
['calendar_year', 'calendar_quarter', 'calendar_month', 'calendar_month_name', 'calendar_week', 'calendar_day', 'calendar_day_name', 'calendar_day_of_week', 'calendar_is_weekend', 'calendar_is_holiday', 'calendar_holiday_name', 'calendar_season', 'store_store_name', 'store_city', 'store_area', 'store_store_type', 'store_opening_date', 'store_seating_capacity', 'store_average_daily_customers', 'store_manager_region', 'product_product_name', 'product_category', 'product_subcategory', 'product_size', 'product_unit_cost', 'product_selling_price', 'product_shelf_life_hours', 'product_preparation_time', 'product_active_flag', 'weather_weather_id', 'weather_temperature', 'weather_rainfall', 'weather_humidity', 'weather_weather_condition', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_

In [23]:
# 20. Feature Data Type Investigation
# Identify numerical and categorical predictors before preprocessing.

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("FEATURE TYPE INVESTIGATION COMPLETED")
print("=" * 50)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical feature columns:")
print(numerical_features)

print("\nCategorical feature columns:")
print(categorical_features)

print("\nTraining feature data types:")
display(X_train.dtypes.to_frame("data_type"))

FEATURE TYPE INVESTIGATION COMPLETED
Numerical features: 22
Categorical features: 18

Numerical feature columns:
['calendar_year', 'calendar_quarter', 'calendar_month', 'calendar_week', 'calendar_day', 'calendar_day_of_week', 'store_seating_capacity', 'store_average_daily_customers', 'product_unit_cost', 'product_selling_price', 'product_shelf_life_hours', 'product_preparation_time', 'weather_temperature', 'weather_rainfall', 'weather_humidity', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28']

Categorical feature columns:
['calendar_month_name', 'calendar_day_name', 'calendar_is_weekend', 'calendar_is_holiday', 'calendar_holiday_name', 'calendar_season', 'store_store_name', 'store_city', 'store_area', 'store_store_type', 'store_manager_region', 'product_product_name', 'product_category', 'product_subcategory', 'product_size', 'product_active_flag', 'weather_weather_id', 'weather_weather_condition']


C:\Users\user\AppData\Local\Temp\ipykernel_10540\2524226828.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


,data_type
calendar_year,int64
calendar_quarter,int64
calendar_month,int64
calendar_month_name,str
calendar_week,int64
calendar_day,int64
calendar_day_name,str
calendar_day_of_week,int64
calendar_is_weekend,bool
calendar_is_holiday,bool


In [25]:
# 21. Model Feature Preparation
# Create useful store operating-day features and remove identifiers.

import importlib
import src.model_preprocessing as mp

importlib.reload(mp)

(
    X_train,
    X_test,
    numerical_features,
    categorical_features
) = mp.prepare_model_features(
    train_data,
    test_data
)

print("MODEL FEATURE PREPARATION COMPLETED")
print("=" * 50)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical columns:")
print(numerical_features)

print("\nCategorical columns:")
print(categorical_features)

2026-11-13 22:52:28,933 | INFO | src.model_preprocessing | Model features prepared | Numerical: 26 | Categorical: 14


MODEL FEATURE PREPARATION COMPLETED
Training features: (7200, 40)
Testing features: (1800, 40)
Numerical features: 26
Categorical features: 14

Numerical columns:
['calendar_year', 'calendar_quarter', 'calendar_month', 'calendar_week', 'calendar_day', 'calendar_day_of_week', 'calendar_is_weekend', 'calendar_is_holiday', 'store_seating_capacity', 'store_average_daily_customers', 'product_unit_cost', 'product_selling_price', 'product_shelf_life_hours', 'product_preparation_time', 'product_active_flag', 'weather_temperature', 'weather_rainfall', 'weather_humidity', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28', 'store_operating_days']

Categorical columns:
['calendar_month_name', 'calendar_day_name', 'calendar_holiday_name', 'calendar_season', 'store_store_name', 'store_city', 'store_area', 'store_store_type', 'store_manager_region', 'product_product_name', 'product_category', 'product_subcategory', '

In [26]:
# 22. Missing Value Investigation
# Inspect missing values in training and testing predictors.

train_missing = X_train.isna().sum()
test_missing = X_test.isna().sum()

missing_report = pd.DataFrame({
    "feature": X_train.columns,
    "data_type": X_train.dtypes.astype(str).values,
    "train_missing": train_missing.values,
    "test_missing": test_missing.values
})

missing_report["total_missing"] = (
    missing_report["train_missing"] +
    missing_report["test_missing"]
)

missing_report = missing_report.sort_values(
    by="total_missing",
    ascending=False
).reset_index(drop=True)

print("MISSING VALUE INVESTIGATION COMPLETED")
print("=" * 50)

print("Training missing values:", int(train_missing.sum()))
print("Testing missing values:", int(test_missing.sum()))

print("\nFeatures containing missing values:")
display(
    missing_report[
        missing_report["total_missing"] > 0
    ]
)

print("\nTotal features with missing values:",
      int((missing_report["total_missing"] > 0).sum()))

MISSING VALUE INVESTIGATION COMPLETED
Training missing values: 12300
Testing missing values: 1600

Features containing missing values:


,feature,data_type,train_missing,test_missing,total_missing
0,calendar_holiday_name,str,7000,1600,8600
1,sales_lag_28,float64,2800,0,2800
2,sales_lag_14,float64,1400,0,1400
3,sales_lag_7,float64,700,0,700
4,sales_rolling_mean_14,float64,100,0,100
5,sales_rolling_mean_28,float64,100,0,100
6,sales_rolling_mean_7,float64,100,0,100
7,sales_lag_1,float64,100,0,100



Total features with missing values: 8


In [28]:
# 23. Missing Value Treatment Preparation
# Replace expected holiday nulls and create indicators
# for unavailable numerical history.

import importlib
import src.model_preprocessing as mp

importlib.reload(mp)

X_train, X_test, missing_indicators = (
    mp.prepare_missing_value_indicators(
        X_train,
        X_test
    )
)

print("MISSING VALUE TREATMENT PREPARATION COMPLETED")
print("=" * 50)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nMissing-value indicators created:")
print(missing_indicators)

print("\nRemaining training missing values:",
      int(X_train.isna().sum().sum()))

print("Remaining testing missing values:",
      int(X_test.isna().sum().sum()))

2026-11-13 22:55:10,652 | INFO | src.model_preprocessing | Missing-value indicators created: 7


MISSING VALUE TREATMENT PREPARATION COMPLETED
Training shape: (7200, 47)
Testing shape: (1800, 47)

Missing-value indicators created:
['sales_lag_1_missing', 'sales_lag_7_missing', 'sales_lag_14_missing', 'sales_lag_28_missing', 'sales_rolling_mean_7_missing', 'sales_rolling_mean_14_missing', 'sales_rolling_mean_28_missing']

Remaining training missing values: 5300
Remaining testing missing values: 0


In [29]:
# 24. Numerical and Categorical Imputation
# Impute missing values using statistics calculated from training data only.

import importlib
import src.model_preprocessing as mp

importlib.reload(mp)

feature_columns = X_train.columns.tolist()

X_train, X_test, imputation_values = (
    mp.impute_missing_features(
        X_train,
        X_test,
        feature_columns
    )
)

print("FEATURE IMPUTATION COMPLETED")
print("=" * 50)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nNumber of imputation values:", len(imputation_values))

print("\nRemaining training missing values:",
      int(X_train.isna().sum().sum()))

print("Remaining testing missing values:",
      int(X_test.isna().sum().sum()))

print("\nImputation values:")
display(
    pd.DataFrame(
        list(imputation_values.items()),
        columns=["feature", "imputation_value"]
    )
)

2026-11-13 22:56:57,430 | INFO | src.model_preprocessing | Feature imputation completed.


FEATURE IMPUTATION COMPLETED
Training shape: (7200, 47)
Testing shape: (1800, 47)

Number of imputation values: 47

Remaining training missing values: 0
Remaining testing missing values: 0

Imputation values:


,feature,imputation_value
0,calendar_year,2025.0
1,calendar_quarter,1.0
2,calendar_month,2.0
3,calendar_month_name,January
4,calendar_week,6.0
5,calendar_day,12.5
6,calendar_day_name,Thursday
7,calendar_day_of_week,4.0
8,calendar_is_weekend,0.0
9,calendar_is_holiday,0.0


In [30]:
# 25. Imputation Validation
# Verify that no missing values remain in training or testing features.

imputation_validation = mp.validate_imputed_features(
    X_train,
    X_test,
    feature_columns
)

print("IMPUTATION VALIDATION COMPLETED")
print("=" * 50)

display(imputation_validation)

print("\nTotal remaining missing values:",
      imputation_validation["missing_count"].sum())

print("\nValidation status:",
      "PASSED" if imputation_validation["missing_count"].sum() == 0
      else "FAILED")

2026-11-13 22:58:00,123 | INFO | src.model_preprocessing | All features passed missing-value validation.


IMPUTATION VALIDATION COMPLETED


,dataset,feature,missing_count,data_type
0,Training,calendar_year,0,int64
1,Training,calendar_quarter,0,int64
2,Training,calendar_month,0,int64
3,Training,calendar_month_name,0,str
4,Training,calendar_week,0,int64
...,...,...,...,...
89,Testing,sales_lag_14_missing,0,int64
90,Testing,sales_lag_28_missing,0,int64
91,Testing,sales_rolling_mean_7_missing,0,int64
92,Testing,sales_rolling_mean_14_missing,0,int64



Total remaining missing values: 0

Validation status: PASSED


In [31]:
# 26. Categorical Feature Cardinality Analysis
# Examine unique categories before encoding categorical predictors.

categorical_cardinality = pd.DataFrame({
    "feature": categorical_features,
    "unique_categories": [
        X_train[column].nunique()
        for column in categorical_features
    ],
    "data_type": [
        str(X_train[column].dtype)
        for column in categorical_features
    ]
})

categorical_cardinality = categorical_cardinality.sort_values(
    by="unique_categories",
    ascending=False
).reset_index(drop=True)

print("CATEGORICAL CARDINALITY ANALYSIS COMPLETED")
print("=" * 50)

display(categorical_cardinality)

print("\nTotal categorical features:",
      len(categorical_features))

print("Total unique category combinations:",
      categorical_cardinality["unique_categories"].sum())

CATEGORICAL CARDINALITY ANALYSIS COMPLETED


,feature,unique_categories,data_type
0,product_product_name,20,str
1,product_subcategory,9,str
2,calendar_day_name,7,str
3,store_manager_region,5,str
4,store_store_name,5,str
5,store_city,5,str
6,store_area,5,str
7,weather_weather_condition,4,str
8,calendar_month_name,3,str
9,calendar_holiday_name,3,str



Total categorical features: 14
Total unique category combinations: 77


In [35]:
import importlib
import src.model_preprocessing as mp

importlib.reload(mp)

<module 'src.model_preprocessing' from 'd:\\Data Analysis Projects\\SmartServe-Food-Intelligence\\src\\model_preprocessing.py'>

In [34]:
# 27. Categorical Feature Encoding
# Apply one-hot encoding using training categories only.

import importlib
import src.model_preprocessing as mp

importlib.reload(mp)

X_train, X_test, categorical_encoder = (
    mp.encode_categorical_features(
        X_train,
        X_test,
        categorical_features
    )
)

print("CATEGORICAL ENCODING COMPLETED")
print("=" * 50)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nOriginal categorical features:",
      len(categorical_features))

print("Generated encoded features:",
      len(categorical_encoder.get_feature_names_out(
          categorical_features
      )))

print("\nRemaining non-numerical training columns:")
print(X_train.select_dtypes(exclude=["number"]).columns.tolist())

2026-11-13 23:01:34,098 | INFO | src.model_preprocessing | Categorical encoding completed | Encoded columns: 77


CATEGORICAL ENCODING COMPLETED
Training shape: (7200, 110)
Testing shape: (1800, 110)

Original categorical features: 14
Generated encoded features: 77

Remaining non-numerical training columns:
['calendar_is_weekend', 'calendar_is_holiday', 'product_active_flag']


In [36]:
# 28. Boolean Feature Conversion
# Convert remaining Boolean predictors into numerical values.

import importlib
import src.model_preprocessing as mp

importlib.reload(mp)

X_train, X_test, boolean_columns = (
    mp.convert_boolean_features(
        X_train,
        X_test
    )
)

print("BOOLEAN FEATURE CONVERSION COMPLETED")
print("=" * 50)

print("Converted columns:", boolean_columns)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nRemaining non-numerical columns:")
print(
    X_train.select_dtypes(
        exclude=["number"]
    ).columns.tolist()
)

2026-11-13 23:03:13,459 | INFO | src.model_preprocessing | Boolean features converted: 3


BOOLEAN FEATURE CONVERSION COMPLETED
Converted columns: ['calendar_is_weekend', 'calendar_is_holiday', 'product_active_flag']

Training shape: (7200, 110)
Testing shape: (1800, 110)

Remaining non-numerical columns:
[]


In [37]:
# 29. Final Model Dataset Validation
# Validate the final numerical training and testing datasets.

print("FINAL MODEL DATASET VALIDATION")
print("=" * 50)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining and testing columns match:",
      X_train.columns.equals(X_test.columns))

print("All training features numerical:",
      all(pd.api.types.is_numeric_dtype(dtype)
          for dtype in X_train.dtypes))

print("All testing features numerical:",
      all(pd.api.types.is_numeric_dtype(dtype)
          for dtype in X_test.dtypes))

print("Training missing values:",
      int(X_train.isna().sum().sum()))

print("Testing missing values:",
      int(X_test.isna().sum().sum()))

print("Training infinite values:",
      int(np.isinf(X_train.select_dtypes(include=["number"])).sum().sum()))

print("Testing infinite values:",
      int(np.isinf(X_test.select_dtypes(include=["number"])).sum().sum()))

print("\nTarget dimensions:")
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

FINAL MODEL DATASET VALIDATION
Training shape: (7200, 110)
Testing shape: (1800, 110)

Training and testing columns match: True
All training features numerical: True
All testing features numerical: True
Training missing values: 0
Testing missing values: 0
Training infinite values: 0
Testing infinite values: 0

Target dimensions:
y_train: (7200,)
y_test: (1800,)


In [39]:
# 30. Save Processed Model Datasets
# Save training and testing features and targets as CSV files.

from pathlib import Path

PROCESSED_MODEL_DIR = PROJECT_ROOT / "data" / "processed" / "model"

PROCESSED_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

X_train.to_csv(
    PROCESSED_MODEL_DIR / "X_train.csv",
    index=False
)

X_test.to_csv(
    PROCESSED_MODEL_DIR / "X_test.csv",
    index=False
)

y_train.to_frame(name=TARGET_COLUMN).to_csv(
    PROCESSED_MODEL_DIR / "y_train.csv",
    index=False
)

y_test.to_frame(name=TARGET_COLUMN).to_csv(
    PROCESSED_MODEL_DIR / "y_test.csv",
    index=False
)

print("PROCESSED MODEL DATASETS SAVED")
print("=" * 50)

print("Output directory:", PROCESSED_MODEL_DIR)

for file_path in sorted(PROCESSED_MODEL_DIR.glob("*.csv")):
    print(
        file_path.name,
        "|",
        file_path.stat().st_size,
        "bytes"
    )

PROCESSED MODEL DATASETS SAVED
Output directory: d:\Data Analysis Projects\SmartServe-Food-Intelligence\data\processed\model
X_test.csv | 817630 bytes
X_train.csv | 3228759 bytes
y_test.csv | 7183 bytes
y_train.csv | 28708 bytes


In [40]:
# 31. Verify Saved Model Datasets
# Reload saved files and compare them with the original datasets.

saved_X_train = pd.read_csv(
    PROCESSED_MODEL_DIR / "X_train.csv"
)

saved_X_test = pd.read_csv(
    PROCESSED_MODEL_DIR / "X_test.csv"
)

saved_y_train = pd.read_csv(
    PROCESSED_MODEL_DIR / "y_train.csv"
)

saved_y_test = pd.read_csv(
    PROCESSED_MODEL_DIR / "y_test.csv"
)

print("SAVED DATASET VERIFICATION")
print("=" * 50)

print("X_train shape:", saved_X_train.shape)
print("X_test shape:", saved_X_test.shape)
print("y_train shape:", saved_y_train.shape)
print("y_test shape:", saved_y_test.shape)

print("\nTraining features match:",
      saved_X_train.equals(X_train.reset_index(drop=True)))

print("Testing features match:",
      saved_X_test.equals(X_test.reset_index(drop=True)))

print("Training target match:",
      saved_y_train[TARGET_COLUMN].equals(y_train.reset_index(drop=True)))

print("Testing target match:",
      saved_y_test[TARGET_COLUMN].equals(y_test.reset_index(drop=True)))

SAVED DATASET VERIFICATION
X_train shape: (7200, 110)
X_test shape: (1800, 110)
y_train shape: (7200, 1)
y_test shape: (1800, 1)

Training features match: False
Testing features match: False
Training target match: True
Testing target match: True


In [41]:
# 32. Investigate Saved Feature Differences
# Check whether CSV reloading changed values or data types.

print("TRAINING DATA COMPARISON")
print("=" * 50)

print("Original dtypes:")
print(X_train.dtypes.value_counts())

print("\nReloaded dtypes:")
print(saved_X_train.dtypes.value_counts())

print("\nMaximum absolute numerical difference:")

train_difference = (
    X_train.reset_index(drop=True) - saved_X_train
).abs()

test_difference = (
    X_test.reset_index(drop=True) - saved_X_test
).abs()

print(
    "Training:",
    train_difference.select_dtypes(include=["number"]).max().max()
)

print(
    "Testing:",
    test_difference.select_dtypes(include=["number"]).max().max()
)

print("\nExact column match:")
print(X_train.columns.equals(saved_X_train.columns))
print(X_test.columns.equals(saved_X_test.columns))

TRAINING DATA COMPARISON
Original dtypes:
float64    86
int64      24
Name: count, dtype: int64

Reloaded dtypes:
float64    86
int64      24
Name: count, dtype: int64

Maximum absolute numerical difference:
Training: 7.105427357601002e-15
Testing: 7.105427357601002e-15

Exact column match:
True
True


In [42]:
# 33. Final Saved Dataset Verification
# Allow negligible floating-point differences after CSV reloading.

import numpy as np

print("FINAL SAVED DATASET VERIFICATION")
print("=" * 50)

print(
    "Training features verified:",
    np.allclose(
        X_train.to_numpy(),
        saved_X_train.to_numpy(),
        rtol=1e-12,
        atol=1e-12
    )
)

print(
    "Testing features verified:",
    np.allclose(
        X_test.to_numpy(),
        saved_X_test.to_numpy(),
        rtol=1e-12,
        atol=1e-12
    )
)

print(
    "Training target verified:",
    y_train.reset_index(drop=True).equals(
        saved_y_train[TARGET_COLUMN]
    )
)

print(
    "Testing target verified:",
    y_test.reset_index(drop=True).equals(
        saved_y_test[TARGET_COLUMN]
    )
)

FINAL SAVED DATASET VERIFICATION
Training features verified: True
Testing features verified: True
Training target verified: True
Testing target verified: True
